In [3]:
import os,json,time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY=os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b")

c:\Users\bhanu\Desktop\AI Agents course\AI-Agents-Learning\AgenticAIWorkspace\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from pageindex import PageIndexClient
pi_client=PageIndexClient(api_key=PAGEINDEX_API_KEY)

## Upload & Index a PDF

### What happens here:

1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchial <b> tree index </b> like a smart table of contents
4. Returns a doc_id for all future operations

### Why no chunking?
Instead of cutting the document into arbitary 500-token pieces, PageIndex respects the document's natural section boundaries -- chapters, sub-sections, paragraphs -- as the author intended

In [7]:
pdf_path ="../Bhanu final.pdf"
print("Uploading PDF to PageIndex...")
result=pi_client.submit_document(pdf_path)
doc_id=result["doc_id"]
print(f"Document id: {doc_id}")

Uploading PDF to PageIndex...
Document id: pi-cmqag7pn3001v01pk7fvky611


In [8]:
print("Building tree index...")
print("This runs once per document -- the index is cached for reuse")

while True:
    status_result=pi_client.get_document(doc_id)
    status=status_result.get("status")
    print(f"Status: {status}")

    if status == "completed":
        print("\nIndexing completed!")
        break
    elif status == "failed":
        print("\nIndexing failed. Please check the PageIndex dashboard for more details.")
        break

    time.sleep(5)

Building tree index...
This runs once per document -- the index is cached for reuse
Status: completed

Indexing completed!


## Inspect the Tree Structure

Each node has:
1. note_id -- unique ID used during retrieval
2. title -- section heading
3. page_index -- page number in original pdf
4. text -- section summary (when node_summary=True)
5. nodes -- child sections (nested)

<b> This structure is what the LLM reasons over during retrieval </b>

In [9]:
tree_result= pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree=tree_result.get("result",[])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\n Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 9

 Raw tree (first node):
{
  "title": "A Project report on",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# A Project report on\n",
  "text": "# A Project report on\n"
}


In [10]:
def print_tree(nodes,indent=0):
    """Recursively print tree titles for a visual overview"""
    for node in nodes:
        prefix = "  "* indent + ("|__" if indent > 0 else "")
        page=node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent +1)

print(" Full Ducument Structure: \n")
print_tree(pageindex_tree)

 Full Ducument Structure: 

[0000] A Project report on (p.1)
[0001] Fraud Detection in Banking Data by Machine Learning Techniques (p.1)
[0002] Bachelor of Technology (p.1)
  |__[0003] Computer Science and Engineering (AI&amp;ML) (p.1)
    |__[0004] Department of Computer Science and Engineering (AI&amp;ML) (p.1)
    |__[0005] CMR COLLEGE OF ENGINEERING &amp; TECHNOLOGY (p.1)
      |__[0006] ACKNOWLEDGEMENT (p.3)
      |__[0007] List of Tables (p.7)
      |__[0008] ABSTRACT (p.8)
[0009] CHAPTER 1 INTRODUCTION (p.9)
[0010] CHAPTER 2 (p.13)
[0011] CHAPTER 3 PROPOSED SYSTEM (p.19)
  |__[0012] 3.1 Research Objective of Proposed Model (p.20)
  |__[0013] Introduction (p.20)
  |__[0014] Objective of the Proposed Model (p.20)
  |__[0015] Key Focus Areas (p.20)
  |__[0016] Supervised Learning Algorithms (p.22)
  |__[0017] Unsupervised Learning Algorithms (p.22)
  |__[0018] Anomaly Detection Techniques (p.22)
  |__[0019] 3.3 Designing (p.22)
  |__[0020] 3.4 UML Diagrams: (p.23)
  |__[0021] 3.5 I

In [11]:
def count_nodes(nodes):
    total=len(nodes)
    for n in nodes:
        if n.get("nodes"):
                 total+=count_nodes(n["nodes"])
    return total

total=count_nodes(pageindex_tree)
print(f" Total nodes in tree: {total}")
print("     Each node = one retrievable section of the document")

 Total nodes in tree: 40
     Each node = one retrievable section of the document


## LLM Tree Search The Core of Pagelndex

This is where Pagelndex fundamentally differs from vector RAG.

### Vector RAG retrieval:

query -> embed -> cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks

Problem: finds what's similar, not what's relevant

### PageIndex retrieval:

query + tree -> LLM reasons "node 0007 and 0008 contain the answer"

Advantage: LLM understands document structure, context, and intent

<b>The LLM acts like a human expert scanning a Table of Contents.</b>

In [12]:
from pydantic import BaseModel, Field
from typing import List

class TreeSearchResponse(BaseModel):
    thinking: str = Field(description="Reasoning about relevant sections")
    node_list: List[str] = Field(description="Relevant node IDs")

In [13]:
structured_llm = llm.with_structured_output(TreeSearchResponse)

In [ ]:
### LLM Tree search function

from urllib import response


def llm_tree_search(query: str, tree:list, llm)-> dict:
    """
    Core PageIndex retrieval:
    Sens the query + document tree to an LLM.
    LLm reasons over the structure and returns relevant node_ids.

    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    def compress(nodes):
        out=[]
        for n in nodes:
            entry={
                "node_id":n["node_id"],
                "title":n["title"],
                "page":n.get("page_index", "?"),
                "summary":n.get("text", "")[:150] # first 150 chars
            }
            if n.get("nodes"):
                entry["children"]=compress(n["nodes"])
            out.append(entry)
        return out
    compressed_tree=compress(tree)

    prompt=f"""You are given a query and a document's tree structure (like a Table of Contents).
    Your task: identify which node IDs most likely contain the answer to the query.
    Think step-by-step about which sections are relevant.
    
    Query: {query}
    Document Tree:
    {json.dumps(compressed_tree,indent=2)}
"""
    response = structured_llm.invoke(prompt)
    return response.model_dump()

In [17]:
#Test with a sample query:
query="What is the project about?"

print(f"Query: {query}\n")
result=llm_tree_search(query,pageindex_tree,structured_llm)

print("LLM Reasoning:")
print(result.get("thinking","N/A"))
print()
print("Selected Node IDs:",result.get("node_list",[]))

Query: What is the project about?

LLM Reasoning:
The question asks for the project’s purpose or overview. The abstract (node 0008) usually gives a concise summary of the project, and the introduction (node 0010) provides a detailed explanation. These sections are most likely to contain the answer.

Selected Node IDs: ['0008', '0010']


## Full RAG Pipeline:

### 3 steps:

1. <b> Tree Search</b> -> LLM picks relevant node_ids
2. <b> Retrieve</b> -> Fetch the actual section content from those nodes
3. <b>Generate</b> _> LLM writes a grounded answer with page citations

### What makes this better than vector RAG:

1. Retrieved content has titles + page numbers (traceable)
2. LLM can cite exactly which section the answer comes from
3. No hallucination from irrevevant chunks

In [ ]:
#Helper: Find nodes by ID

def find_nodes_by_ids(tree:list, target_ids:list)->list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found=[]
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [19]:
# Generate answer from retrieved nodes

def generate_answer(query:str, nodes:list,llm)->str:
    """
    Takes retreieved nodes as context and generated a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "No relevant sections found in the document."
    
    #Build context string from retrieved nodes
    context_parts=[]
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index','?')}]\n"
            f"{node.get('text','Content not available')}"
        )
    context="\n\n---\n\n".join(context_parts)

    prompt=f"""You are an expert document analyst.
    answer the question using ONLY the provided context.
    For every claim you make, cite the section title and page number in paranthesis.
    Be consice and precise.
    
    Question: {query}
    Context:
    {context}

    Answer:"""

    response = llm.invoke(prompt)
    return response.content

In [22]:
def vectorless_rag(query:str,tree:list,verbose:bool=True)->str:
    """
    Full end-to-edn PageIndex RAG pipeline:

    step-1: LLM Tree Search -> finds relevant node_ids
    step-2: Node Retrieval -> fetches section content
    step-3: Answer Generation -> produces cited answer
    """

    if verbose:
        print(f"{'='*55}")
        print(f"Query:{query}")
        print(f"{'='*55}")

    # Step-1: Tree Search
    search_result=llm_tree_search(query, tree, structured_llm)
    node_ids=search_result.get("node_list",[])

    if verbose:
        print(f"\n Reasoning; {search_result.get('thinking','')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")

    #Step-2: Node Retrieval
    nodes=find_nodes_by_ids(tree,node_ids)

    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")

    #Step-3: Generate answer
    answer=generate_answer(query,nodes,llm)

    if verbose:
        print(f"\n Answer:\n{answer}")
    
    return answer

In [23]:
# Run the full pipeline with our sample query
answer=vectorless_rag(
    query="What are the algorithms used in the project?",
    tree=pageindex_tree,
)

Query:What are the algorithms used in the project?

 Reasoning; I identified that the sections describing supervised, unsupervised, anomaly detection algorithms (nodes 0016-0018) and the implementation code sections (nodes 0021-0024) contain the algorithms used in...
Retrieved node IDs: ['0016', '0017', '0018', '0021', '0022', '0023', '0024']
Sections found: ['Supervised Learning Algorithms', 'Unsupervised Learning Algorithms', 'Anomaly Detection Techniques', '3.5 IMPLEMENTATION CODE', 'Implementation of Machine Learning Models and Data Processing', 'Model Implementation, Evaluation, and Visualization', 'System Implementation and Graphical User Interface Development']

 Answer:
**Algorithms employed in the project**

| Algorithm | Category | Source (Section title & page) |
|-----------|----------|--------------------------------|
| Logistic Regression | Supervised | “Supervised Learning Algorithms” (Page 22) |
| Decision Tree | Supervised | “Supervised Learning Algorithms” (Page 22) |
